# Information Retrieval — Term 481 — Assignment 1
## Boolean Retrieval, Indexing & Preprocessing

**Based on:** Topic 1 (*Introduction to IR* + Boolean Retrieval) and Lecture 2 / Topic 2 (*Indexing & Preprocessing*).

### Instructions
- Work through each exercise in order. Each exercise has a **"Your Code"** cell for you to complete.


---
## Part 1 — Term–Document Incidence Matrix & Boolean Retrieval

We reuse the idea from the Lecture 1 notebook but with a **new** collection of 6 short English documents:


In [85]:
collection = {
    "d1": "the cat sat on the mat",
    "d2": "the dog sat on the log",
    "d3": "the cat chased the dog",
    "d4": "a mat is not a log",
    "d5": "the dog and the cat are friends",
    "d6": "cats and dogs sat together",
}
collection

{'d1': 'the cat sat on the mat',
 'd2': 'the dog sat on the log',
 'd3': 'the cat chased the dog',
 'd4': 'a mat is not a log',
 'd5': 'the dog and the cat are friends',
 'd6': 'cats and dogs sat together'}

### Exercise 1.1 — Build the Term–Document Incidence Matrix by hand (2 points)

Before writing any code, fill in the table below (replace the `?` with `1` if the term appears in the document, `0` otherwise). Use the **raw** words as they appear (no stemming yet) — do NOT split "cats"/"cat" or "dogs"/"dog".

| term | d1 | d2 | d3 | d4 | d5 | d6 |
|---|---|---|---|---|---|---|
| the | 1 | 1 | 1 | 0 | 1 | 0 |
| cat | 1 | 0 | 1 | 0 | 1 | 0 |
| sat | 1 | 1 | 0 | 0 | 0 | 1 |
| on | 1 | 1 | 0 | 0 | 0 | 0 |
| mat | 1 | 0 | 0 | 1 | 0 | 0 |
| dog | 0 | 1 | 1 | 0 | 1 | 0 |
| log | 0 | 1 | 0 | 1 | 0 | 0 |
| chased | 0 | 0 | 1 | 0 | 0 | 0 |
| a | 0 | 0 | 0 | 1 | 0 | 0 |
| is | 0 | 0 | 0 | 1 | 0 | 0 |
| not | 0 | 0 | 0 | 1 | 0 | 0 |
| and | 0 | 0 | 0 | 0 | 1 | 1 |
| are | 0 | 0 | 0 | 0 | 1 | 0 |
| friends | 0 | 0 | 0 | 0 | 1 | 0 |
| cats | 0 | 0 | 0 | 0 | 0 | 1 |
| dogs | 0 | 0 | 0 | 0 | 0 | 1 |
| together | 0 | 0 | 0 | 0 | 0 | 1 |

### Exercise 1.2 — Implement the matrix builder in code (5 points)

Fill in the three helper functions below (mirroring the ones used in the Lecture 1 notebook):
- `get_terms(collection)` → list of **all** term occurrences (with repetition) across the collection.
- `get_unique_terms(terms)` → list of terms with duplicates removed (keep first-seen order).
- `term_document_incidence_matrix(collection)` → dict mapping each unique term to its 0/1 incidence vector, one entry per document (in the collection's iteration order).


In [86]:
def get_terms(collection):
    """All term occurrences (with repetition) across the collection."""
    terms = []
    for doc in collection.values():
        terms.extend(doc.split())
    return terms

def get_unique_terms(terms):
    """Remove duplicates, keeping first-seen order."""
    seen = set()
    unique = []
    for t in terms:
        if t not in seen:
            seen.add(t)
            unique.append(t)
    return unique

def term_document_incidence_matrix(collection):
    """term -> [0/1 per document, in the collection's iteration order]."""
    unique_terms = get_unique_terms(get_terms(collection))
    doc_words = [set(doc.split()) for doc in collection.values()]
    return {t: [1 if t in words else 0 for words in doc_words] for t in unique_terms}

In [87]:
# The output generated by the code written to answer this question should be as follows:

matrix = term_document_incidence_matrix(collection)
print("term".ljust(10), *collection.keys())
for term, vec in matrix.items():
    print(term.ljust(10), *[" " + str(v) for v in vec])

term       d1 d2 d3 d4 d5 d6
the         1  1  1  0  1  0
cat         1  0  1  0  1  0
sat         1  1  0  0  0  1
on          1  1  0  0  0  0
mat         1  0  0  1  0  0
dog         0  1  1  0  1  0
log         0  1  0  1  0  0
chased      0  0  1  0  0  0
a           0  0  0  1  0  0
is          0  0  0  1  0  0
not         0  0  0  1  0  0
and         0  0  0  0  1  1
are         0  0  0  0  1  0
friends     0  0  0  0  1  0
cats        0  0  0  0  0  1
dogs        0  0  0  0  0  1
together    0  0  0  0  0  1


### Exercise 1.3 — Boolean Retrieval (5 points)

Implement `boolean_retrieval(query, collection)` supporting `AND`, `OR`, and `NOT`, evaluated **left to right** (no operator precedence needed), following the same design as the Lecture 1 notebook. Then answer the three queries below:

1. `cat AND dog`
2. `mat OR log`
3. `dog AND NOT cat`


In [88]:
def boolean_operator_processing(op, v1, v2=None):
    """Apply a Boolean operator on 0/1 incidence vectors."""
    if op == "NOT":
        return [1 - a for a in v1]
    if op == "AND":
        return [a & b for a, b in zip(v1, v2)]
    if op == "OR":
        return [a | b for a, b in zip(v1, v2)]
    raise ValueError("Unknown operator: " + op)

def boolean_retrieval(query, collection):
    """Evaluate AND / OR / NOT left to right (no precedence).
    NOT is unary and applies to the term right after it."""
    matrix = term_document_incidence_matrix(collection)
    n_docs = len(collection)
    doc_ids = list(collection.keys())

    def term_vector(term):
        # a term that is not in the collection matches nothing
        return matrix.get(term.lower(), [0] * n_docs)

    tokens = query.split()
    i = 0
    result = None
    pending_op = None
    while i < len(tokens):
        tok = tokens[i]
        if tok in ("AND", "OR"):
            pending_op = tok
            i += 1
            continue
        # operand (possibly negated)
        if tok == "NOT":
            vec = boolean_operator_processing("NOT", term_vector(tokens[i + 1]))
            i += 2
        else:
            vec = term_vector(tok)
            i += 1
        if result is None:
            result = vec
        else:
            result = boolean_operator_processing(pending_op, result, vec)
    return [doc_ids[k] for k, bit in enumerate(result) if bit == 1]

In [89]:
# The output generated by the code written to answer this question should be as follows:

for q in ["cat AND dog", "mat OR log", "dog AND NOT cat"]:
    print(f"{q:<18} -> {boolean_retrieval(q, collection)}")

cat AND dog        -> ['d3', 'd5']
mat OR log         -> ['d1', 'd2', 'd4']
dog AND NOT cat    -> ['d2']


---
## Part 2 — Inverted Index Construction & Query Processing

Recall the 3-step construction method from Topic 2 (Indexing & Preprocessing):

1. **Term sequence** — build a list of `(term, docID)` pairs.
2. **Sort** — sort the pairs by term, then by docID.
3. **Dictionary & Postings** — group by term into a *postings list* of docIDs (with document frequency `df`).

We'll build it for the two mini-documents:

> Doc 1: *"I did enact Julius Caesar I was killed i' the Capitol Brutus killed me"*
>
> Doc 2: *"So let it be with Caesar The noble Brutus hath told you Caesar was ambitious"*


In [90]:
docs = {
    1: "I did enact Julius Caesar I was killed i the Capitol Brutus killed me",
    2: "So let it be with Caesar The noble Brutus hath told you Caesar was ambitious",
}

### Exercise 2.1 — Build the inverted index (5 points)

Implement `build_inverted_index(docs)` that:
1. Lowercases every token,
2. Builds the sorted `(term, docID)` sequence,
3. Groups it into a dictionary: `term -> {"df": <#docs containing term>, "postings": [sorted list of docIDs, one entry per occurrence]}`.

(For simplicity, keep one posting per *occurrence*, i.e. `caesar` in doc 2 — which occurs twice — should appear twice in its postings list, matching the slide's "Core indexing step" table.)


In [91]:
def build_inverted_index(docs):
    # 1) term sequence: (term, docID) pairs, lower-cased
    pairs = []
    for doc_id, text in docs.items():
        for token in text.lower().split():
            pairs.append((token, doc_id))
    # 2) sort by term, then docID
    pairs.sort()
    # 3) group into dictionary + postings (one posting per occurrence)
    index = {}
    for term, doc_id in pairs:
        if term not in index:
            index[term] = {"df": 0, "postings": []}
        index[term]["postings"].append(doc_id)
    for term in index:
        index[term]["df"] = len(set(index[term]["postings"]))
    return index

In [92]:
# The output generated by the code written to answer this question should be as follows:

inv_index = build_inverted_index(docs)
for term, entry in inv_index.items():
    print(f"{term:<10} df={entry['df']}  postings={entry['postings']}")

ambitious  df=1  postings=[2]
be         df=1  postings=[2]
brutus     df=2  postings=[1, 2]
caesar     df=2  postings=[1, 2, 2]
capitol    df=1  postings=[1]
did        df=1  postings=[1]
enact      df=1  postings=[1]
hath       df=1  postings=[2]
i          df=1  postings=[1, 1, 1]
it         df=1  postings=[2]
julius     df=1  postings=[1]
killed     df=1  postings=[1, 1]
let        df=1  postings=[2]
me         df=1  postings=[1]
noble      df=1  postings=[2]
so         df=1  postings=[2]
the        df=2  postings=[1, 2]
told       df=1  postings=[2]
was        df=2  postings=[1, 2]
with       df=1  postings=[2]
you        df=1  postings=[2]


### Exercise 2.2 — AND merge algorithm (5 points)

Implement `and_merge(p1, p2)` that intersects **two sorted, duplicate-free** postings lists in `O(len(p1) + len(p2))` — i.e., use the two-pointer "merge" technique shown in the slides, **not** Python's `set` intersection (which would defeat the purpose of the exercise, even though it gives the same result).

Test it on:
```
wink:  [1, 2, 4, 8, 16, 32, 64]
drink: [1, 2, 3, 5, 8, 13, 21, 34]
```


In [93]:
def and_merge(p1, p2):
    """Intersection of two sorted, duplicate-free lists in O(len(p1)+len(p2))."""
    i = j = 0
    answer = []
    while i < len(p1) and j < len(p2):
        if p1[i] == p2[j]:
            answer.append(p1[i])
            i += 1
            j += 1
        elif p1[i] < p2[j]:
            i += 1
        else:
            j += 1
    return answer

In [94]:
# The output generated by the code written to answer this question should be as follows:

wink  = [1, 2, 4, 8, 16, 32, 64]
drink = [1, 2, 3, 5, 8, 13, 21, 34]
print("wink AND drink =", and_merge(wink, drink))

wink AND drink = [1, 2, 8]


### Exercise 2.3 — OR merge algorithm (5 points)

Now implement `or_merge(p1, p2)` (union of two sorted, duplicate-free lists), also with the two-pointer technique (`O(len(p1)+len(p2))`, no sets).


In [95]:
def or_merge(p1, p2):
    """Union of two sorted, duplicate-free lists in O(len(p1)+len(p2))."""
    i = j = 0
    answer = []
    while i < len(p1) and j < len(p2):
        if p1[i] == p2[j]:
            answer.append(p1[i])
            i += 1
            j += 1
        elif p1[i] < p2[j]:
            answer.append(p1[i])
            i += 1
        else:
            answer.append(p2[j])
            j += 1
    answer.extend(p1[i:])   # leftovers (only one of these is non-empty)
    answer.extend(p2[j:])
    return answer

In [96]:
# The output generated by the code written to answer this question should be as follows:

wink  = [1, 2, 4, 8, 16, 32, 64]
drink = [1, 2, 3, 5, 8, 13, 21, 34]
print("wink OR drink =", or_merge(wink, drink))

wink OR drink = [1, 2, 3, 4, 5, 8, 13, 16, 21, 32, 34, 64]


### Exercise 2.4 — Answer with the real inverted index (5 points)

Using `inv_index` from Exercise 2.1 (and `set()` to drop duplicate docIDs from postings before merging), answer: **`caesar AND brutus`** and **`caesar OR ambitious`**.


In [97]:
def postings_of(term):
    # set() drops the duplicate docIDs (one posting per occurrence), sorted() restores order
    return sorted(set(inv_index[term]["postings"]))

print("caesar AND brutus   :", and_merge(postings_of("caesar"), postings_of("brutus")))
print("caesar OR ambitious :", or_merge(postings_of("caesar"), postings_of("ambitious")))

caesar AND brutus   : [1, 2]
caesar OR ambitious : [1, 2]


In [98]:
# The output generated by the code written to answer this question should be as follows:

print("postings(caesar)    =", postings_of("caesar"))
print("postings(brutus)    =", postings_of("brutus"))
print("postings(ambitious) =", postings_of("ambitious"))
print("caesar AND brutus   =", and_merge(postings_of("caesar"), postings_of("brutus")))
print("caesar OR ambitious =", or_merge(postings_of("caesar"), postings_of("ambitious")))

postings(caesar)    = [1, 2]
postings(brutus)    = [1, 2]
postings(ambitious) = [2]
caesar AND brutus   = [1, 2]
caesar OR ambitious = [1, 2]


---
## Part 3 — Text Preprocessing Pipeline (Tokenization → Stopping → Stemming)

We will build the full pipeline shown in the Topic 2 slides:

> Input: *"This is an example sentence of how the pre-processing is applied to text in information retrieval. It includes: Tokenization, Stop Words Removal, and Stemming"*
>
> Expected output: `exampl sentenc pre process appli text inform retriev includ token stop word remov stem`

This exercise uses `nltk`'s Porter Stemmer (the classic algorithm covered in the slides).


In [99]:
import re
from nltk.stem import PorterStemmer

STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were", "this", "that", "of", "to",
    "in", "on", "for", "and", "it", "how", "with", "as", "by", "at", "from",
}

### Exercise 3.1 — Tokenization (5 points)

Implement `tokenize(text)` that:
- lower-cases the text,
- splits on any run of non-letter characters (so punctuation, digits, and hyphens act as separators — following the *"just split at non-letter characters"* common practice from the slides).


In [100]:
def tokenize(text):
    # lower-case, then split on any run of non-letter characters
    return [t for t in re.split(r"[^a-z]+", text.lower()) if t]

In [101]:
# The output generated by the code written to answer this question should be as follows:

text = "This is an example sentence of how the pre-processing is applied to text in information retrieval. It includes: Tokenization, Stop Words Removal, and Stemming"
tokens = tokenize(text)
print(tokens)

['this', 'is', 'an', 'example', 'sentence', 'of', 'how', 'the', 'pre', 'processing', 'is', 'applied', 'to', 'text', 'in', 'information', 'retrieval', 'it', 'includes', 'tokenization', 'stop', 'words', 'removal', 'and', 'stemming']


### Exercise 3.2 — Stop-word Removal (5 points)

Implement `remove_stopwords(tokens)` using the `STOPWORDS` set defined above.


In [102]:
def remove_stopwords(tokens):
    return [t for t in tokens if t not in STOPWORDS]

In [103]:
# The output generated by the code written to answer this question should be as follows:

no_stop = remove_stopwords(tokens)
print(no_stop)

['example', 'sentence', 'pre', 'processing', 'applied', 'text', 'information', 'retrieval', 'includes', 'tokenization', 'stop', 'words', 'removal', 'stemming']


### Exercise 3.3 — Stemming (5 points)

Implement `stem_tokens(tokens)` using `nltk.stem.PorterStemmer`.


In [104]:
stemmer = PorterStemmer()

def stem_tokens(tokens):
    return [stemmer.stem(t) for t in tokens]

In [105]:
# The output generated by the code written to answer this question should be as follows:

stemmed = stem_tokens(no_stop)
print(stemmed)

['exampl', 'sentenc', 'pre', 'process', 'appli', 'text', 'inform', 'retriev', 'includ', 'token', 'stop', 'word', 'remov', 'stem']


### Exercise 3.4 — Full Pipeline (3 points)

Combine 3.1–3.3 into a single `preprocess(text)` function and verify it reproduces the golden output exactly.


In [106]:
def preprocess(text):
    return stem_tokens(remove_stopwords(tokenize(text)))

In [107]:
# The output generated by the code written to answer this question should be as follows:

result = " ".join(preprocess(text))
print(result)

expected = "exampl sentenc pre process appli text inform retriev includ token stop word remov stem"
print("Matches golden output:", result == expected)

exampl sentenc pre process appli text inform retriev includ token stop word remov stem
Matches golden output: True


**Reflection:** if a different stop-word list or stemmer is used, minor differences are expected and normal — this is exactly why preprocessing choices (which stop-word list? which stemmer?) are design decisions that must be documented and applied *consistently* to both documents and queries (as emphasized in the Lecture 2 code notebook, where the *same* `normalize()`/`stem()` functions are re-applied to a search term before looking it up in the index).


---
## Part 4 — Positional Index & Phrase Queries

Recall from the slides: a **positional index** stores, for each term, *which positions* it occurs at within each document (not just *which documents*). This lets us answer **phrase queries** like `"to be"` by checking that two terms occur in the *same* document at *consecutive* positions.


In [108]:
phrase_docs = {
    1: "to be or not to be that is the question",
    2: "I want to be a good friend to be honest",
    3: "being or not being is not the same as to be",
}

### Exercise 4.1 — Build a Positional Index (5 points)

Implement `build_positional_index(docs)` returning `term -> {doc_id: [list of positions (0-based)]}`.


In [109]:
def build_positional_index(docs):
    index = {}
    for doc_id, text in docs.items():
        for pos, token in enumerate(text.lower().split()):
            index.setdefault(token, {}).setdefault(doc_id, []).append(pos)
    return index

In [110]:
# The output generated by the code written to answer this question should be as follows:

pos_index = build_positional_index(phrase_docs)
print("'to' ->", pos_index["to"])
print("'be' ->", pos_index["be"])

'to' -> {1: [0, 4], 2: [2, 7], 3: [9]}
'be' -> {1: [1, 5], 2: [3, 8], 3: [10]}


**Expected:** `'to' -> {1: [0, 4], 2: [2, 7], 3: [9]}` and `'be' -> {1: [1, 5], 2: [3, 8], 3: [10]}`.


### Exercise 4.2 — Phrase Query (5 points)

Implement `phrase_query(pos_index, phrase)` for a **two-word** phrase `"w1 w2"`: it should return the list of docIDs where `w2` occurs at position `p+1` for some occurrence of `w1` at position `p` (i.e. the words are adjacent, in order). Test it on `"to be"`.


In [111]:
def phrase_query(pos_index, phrase):
    w1, w2 = phrase.lower().split()
    if w1 not in pos_index or w2 not in pos_index:
        return []
    matches = []
    # only documents containing both words can match
    for doc_id in sorted(set(pos_index[w1]) & set(pos_index[w2])):
        positions_w2 = set(pos_index[w2][doc_id])
        if any(p + 1 in positions_w2 for p in pos_index[w1][doc_id]):
            matches.append(doc_id)
    return matches

In [112]:
# The output generated by the code written to answer this question should be as follows:

print('"to be" ->', phrase_query(pos_index, "to be"))

"to be" -> [1, 2, 3]


Note doc 3 ("... as to be") matches, while a pure Boolean `to AND be` query would also match, say, a hypothetical document containing "be nice to everyone" even though the phrase "to be" never occurs there — this is exactly why phrase queries need position information rather than just document-level term membership.
